# Edge Temporal Smoothing and Event-Level Alerts

This notebook evaluates temporal smoothing for PPE violation alerts. The goal is to reduce transient frame-level false positives by triggering an event only when the same tracked person shows the same violation for at least `N_CONSECUTIVE_FRAMES`.

## 1. Controls

Use detection logs when available. If no suitable logs or videos are attached, the notebook runs a deterministic synthetic sequence to verify the metric pipeline.

In [ ]:
from pathlib import Path

KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')
RESULTS_DIR = KAGGLE_WORKING / 'results'
LOGS_DIR = KAGGLE_WORKING / 'temporal_logs'

MODEL_ONNX_PATH = None
DEVICE = None
FAST_DEBUG = True
SEED = 42
IMGSZ = 640
CONF_THRES = 0.25
N_CONSECUTIVE_FRAMES = 5
TRACK_DISTANCE_PX = 80
MAX_VIDEO_FRAMES = 300 if FAST_DEBUG else 3000
VIDEO_EXTS = {'.mp4', '.avi', '.mov', '.mkv', '.webm'}
VIOLATION_CLASS_IDS = [7, 8, 9, 10]
CLASS_NAMES = {7: 'no_helmet', 8: 'no_goggle', 9: 'no_gloves', 10: 'no_boots'}

for d in [RESULTS_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('N_CONSECUTIVE_FRAMES:', N_CONSECUTIVE_FRAMES)
print('FAST_DEBUG:', FAST_DEBUG)

## 2. Environment and Input Discovery

This section checks packages, GPU visibility, videos, logs, and the baseline ONNX model.

In [ ]:
import json
import math
import random
import subprocess
import sys
from collections import defaultdict, deque

import numpy as np
import pandas as pd

def version_of(import_name):
    try:
        mod = __import__(import_name)
        return getattr(mod, '__version__', 'installed-version-unknown')
    except Exception as exc:
        return f'not installed ({exc.__class__.__name__})'

print('Python:', sys.version)
for name in ['torch', 'ultralytics', 'onnxruntime', 'cv2', 'numpy', 'pandas']:
    print(f'{name}:', version_of(name))

try:
    import torch
    cuda_count = torch.cuda.device_count()
    DEVICE = '0,1' if cuda_count >= 2 else ('0' if cuda_count == 1 else 'cpu') if DEVICE is None else DEVICE
    print('torch.cuda.device_count():', cuda_count)
    for i in range(cuda_count):
        print(f'GPU {i}:', torch.cuda.get_device_name(i))
except Exception as exc:
    DEVICE = DEVICE or 'cpu'
    print('Torch check failed:', repr(exc))

try:
    print(subprocess.check_output(['nvidia-smi'], text=True))
except Exception as exc:
    print('nvidia-smi unavailable:', repr(exc))

def find_files(exts):
    files = []
    for root in [KAGGLE_INPUT, KAGGLE_WORKING, Path.cwd()]:
        if root.exists():
            files.extend([p.resolve() for p in root.rglob('*') if p.is_file() and p.suffix.lower() in exts])
    return sorted(set(files))

def find_model():
    if MODEL_ONNX_PATH is not None:
        return Path(MODEL_ONNX_PATH)
    for root in [KAGGLE_WORKING, KAGGLE_INPUT, Path.cwd()]:
        if not root.exists():
            continue
        for pattern in ['baseline_yolo26n_best.onnx', 'baseline_yolo26n/weights/best.onnx', 'best.onnx']:
            matches = sorted(root.rglob(pattern))
            if matches:
                return matches[0].resolve()
    return None

VIDEO_FILES = find_files(VIDEO_EXTS)[:2]
CSV_LOGS = [p for p in find_files({'.csv'}) if 'event_vs_frame_metrics' not in p.name]
MODEL_ONNX_PATH = find_model()
print('VIDEO_FILES:', VIDEO_FILES)
print('Candidate CSV logs:', CSV_LOGS[:10])
print('MODEL_ONNX_PATH:', MODEL_ONNX_PATH)

## 3. Detection Log Format

Preferred log columns are `video_id`, `frame_id`, `person_id`, `class_id`, `is_pred`, and `is_gt`. If ground truth is absent, the notebook reports pseudo-stability metrics where transient predictions are treated as suppressed false alarms.

In [ ]:
def find_detection_log():
    required = {'frame_id', 'person_id', 'class_id'}
    for path in CSV_LOGS:
        try:
            head = pd.read_csv(path, nrows=5)
            if required.issubset(set(head.columns)):
                return path
        except Exception:
            pass
    return None

def make_synthetic_log():
    rng = random.Random(SEED)
    rows = []
    video_id = 'synthetic_sequence'
    # Stable true no_helmet event for person 1.
    for frame in range(20, 45):
        rows.append({'video_id': video_id, 'frame_id': frame, 'person_id': 1, 'class_id': 7, 'is_gt': 1, 'is_pred': 1})
    # Missed frames create false negatives.
    for frame in [25, 32]:
        rows.append({'video_id': video_id, 'frame_id': frame, 'person_id': 1, 'class_id': 7, 'is_gt': 1, 'is_pred': 0})
    # Transient false positives that temporal smoothing should suppress.
    for frame in [8, 9, 62, 90, 91, 92]:
        rows.append({'video_id': video_id, 'frame_id': frame, 'person_id': 2, 'class_id': 8, 'is_gt': 0, 'is_pred': 1})
    # Stable but false event.
    for frame in range(120, 127):
        rows.append({'video_id': video_id, 'frame_id': frame, 'person_id': 3, 'class_id': 9, 'is_gt': 0, 'is_pred': 1})
    df = pd.DataFrame(rows).sort_values(['video_id', 'frame_id', 'person_id', 'class_id']).reset_index(drop=True)
    path = LOGS_DIR / 'synthetic_detection_log.csv'
    df.to_csv(path, index=False)
    return path

def run_video_inference_log(video_files):
    if not video_files or MODEL_ONNX_PATH is None:
        return None
    try:
        import cv2
        from ultralytics import YOLO
    except Exception as exc:
        print('Video inference unavailable:', repr(exc))
        return None
    model = YOLO(str(MODEL_ONNX_PATH))
    rows = []
    next_track_id = 1
    for video_path in video_files[:2]:
        cap = cv2.VideoCapture(str(video_path))
        frame_id = 0
        tracks = {}
        while cap.isOpened() and frame_id < MAX_VIDEO_FRAMES:
            ok, frame = cap.read()
            if not ok:
                break
            result = model(frame, imgsz=IMGSZ, conf=CONF_THRES, verbose=False, device=DEVICE)[0]
            for box in result.boxes:
                cls_id = int(box.cls[0])
                if cls_id not in VIOLATION_CLASS_IDS:
                    continue
                x1, y1, x2, y2 = [float(v) for v in box.xyxy[0]]
                cx = (x1 + x2) / 2.0
                cy = (y1 + y2) / 2.0
                key = cls_id
                best_id = None
                best_dist = None
                for track_id, (last_cls, last_cx, last_cy, last_frame) in tracks.items():
                    if last_cls != cls_id or frame_id - last_frame > 10:
                        continue
                    dist = ((cx - last_cx) ** 2 + (cy - last_cy) ** 2) ** 0.5
                    if dist <= TRACK_DISTANCE_PX and (best_dist is None or dist < best_dist):
                        best_id = track_id
                        best_dist = dist
                if best_id is None:
                    best_id = next_track_id
                    next_track_id += 1
                tracks[best_id] = (cls_id, cx, cy, frame_id)
                rows.append({
                    'video_id': video_path.name,
                    'frame_id': frame_id,
                    'person_id': best_id,
                    'class_id': cls_id,
                    'confidence': float(box.conf[0]),
                    'is_pred': 1,
                    'is_gt': np.nan,
                })
            frame_id += 1
        cap.release()
    if not rows:
        return None
    path = LOGS_DIR / 'video_inference_detection_log.csv'
    pd.DataFrame(rows).to_csv(path, index=False)
    return path

LOG_PATH = find_detection_log()
if LOG_PATH is None:
    LOG_PATH = run_video_inference_log(VIDEO_FILES)
    if LOG_PATH is not None:
        print('No compatible detection log found. Generated log from video inference:', LOG_PATH)
if LOG_PATH is None:
    LOG_PATH = make_synthetic_log()
    print('No compatible detection log or usable video found. Using synthetic demonstration log:', LOG_PATH)
else:
    print('Using detection log:', LOG_PATH)

detections = pd.read_csv(LOG_PATH)
if 'video_id' not in detections.columns:
    detections['video_id'] = 'video_0'
if 'is_pred' not in detections.columns:
    detections['is_pred'] = 1
if 'is_gt' not in detections.columns:
    detections['is_gt'] = np.nan
detections = detections[detections['class_id'].isin(VIOLATION_CLASS_IDS)].copy()
display(detections.head())

## 4. Optional Video Inference

If test videos are attached and no detection log is provided, this cell can be adapted to run model inference and create detection logs. The default path stays log-based because reproducible event metrics need stable identifiers or annotations.

In [ ]:
print('Video files available:', len(VIDEO_FILES))
print('Current notebook uses log-based evaluation path:', LOG_PATH)

## 5. Temporal Buffer and Event Rule Engine

A predicted event is raised only when a `(video_id, person_id, class_id)` key is positive for at least `N_CONSECUTIVE_FRAMES` consecutive frames.

In [ ]:
def build_intervals(df, flag_col, min_len=1):
    intervals = []
    if df.empty:
        return intervals
    for (video_id, person_id, class_id), group in df[df[flag_col] == 1].groupby(['video_id', 'person_id', 'class_id']):
        frames = sorted(set(int(x) for x in group['frame_id']))
        if not frames:
            continue
        start = prev = frames[0]
        for frame in frames[1:]:
            if frame == prev + 1:
                prev = frame
            else:
                if prev - start + 1 >= min_len:
                    intervals.append({'video_id': video_id, 'person_id': person_id, 'class_id': int(class_id), 'start': start, 'end': prev, 'length': prev - start + 1})
                start = prev = frame
        if prev - start + 1 >= min_len:
            intervals.append({'video_id': video_id, 'person_id': person_id, 'class_id': int(class_id), 'start': start, 'end': prev, 'length': prev - start + 1})
    return intervals

def intervals_overlap(a, b):
    return a['video_id'] == b['video_id'] and a['person_id'] == b['person_id'] and a['class_id'] == b['class_id'] and max(a['start'], b['start']) <= min(a['end'], b['end'])

def score_intervals(preds, gts):
    matched_gt = set()
    tp = 0
    fp = 0
    for pred in preds:
        match = None
        for idx, gt in enumerate(gts):
            if idx in matched_gt:
                continue
            if intervals_overlap(pred, gt):
                match = idx
                break
        if match is None:
            fp += 1
        else:
            tp += 1
            matched_gt.add(match)
    fn = len(gts) - len(matched_gt)
    return tp, fp, fn

def frame_metrics(df):
    if df['is_gt'].isna().all():
        fp = int((df['is_pred'] == 1).sum())
        return {'tp': 0, 'fp': fp, 'fn': 0, 'fpr': 1.0 if fp else 0.0, 'mode': 'pseudo_no_ground_truth'}
    keys = ['video_id', 'frame_id', 'person_id', 'class_id']
    agg = df.groupby(keys).agg(is_pred=('is_pred', 'max'), is_gt=('is_gt', 'max')).reset_index()
    tp = int(((agg['is_pred'] == 1) & (agg['is_gt'] == 1)).sum())
    fp = int(((agg['is_pred'] == 1) & (agg['is_gt'] != 1)).sum())
    fn = int(((agg['is_pred'] != 1) & (agg['is_gt'] == 1)).sum())
    fpr = fp / max(1, fp + tp)
    return {'tp': tp, 'fp': fp, 'fn': fn, 'fpr': fpr, 'mode': 'supervised'}

frame_level = frame_metrics(detections)
gt_intervals = build_intervals(detections, 'is_gt', min_len=1) if not detections['is_gt'].isna().all() else []
pred_frame_intervals = build_intervals(detections, 'is_pred', min_len=1)
pred_event_intervals = build_intervals(detections, 'is_pred', min_len=N_CONSECUTIVE_FRAMES)

if gt_intervals:
    event_tp, event_fp, event_fn = score_intervals(pred_event_intervals, gt_intervals)
    frame_tp, frame_fp, frame_fn = score_intervals(pred_frame_intervals, gt_intervals)
else:
    frame_tp, frame_fp, frame_fn = 0, len(pred_frame_intervals), 0
    event_tp, event_fp, event_fn = 0, len(pred_event_intervals), 0

rows = [
    {'level': 'frame', 'tp': frame_tp if gt_intervals else frame_level['tp'], 'fp': frame_fp if gt_intervals else frame_level['fp'], 'fn': frame_fn if gt_intervals else frame_level['fn']},
    {'level': 'event', 'tp': event_tp, 'fp': event_fp, 'fn': event_fn},
]
metrics_df = pd.DataFrame(rows)
metrics_df['fpr'] = metrics_df['fp'] / (metrics_df['fp'] + metrics_df['tp']).clip(lower=1)
metrics_df['precision'] = metrics_df['tp'] / (metrics_df['tp'] + metrics_df['fp']).clip(lower=1)
metrics_df['recall'] = metrics_df['tp'] / (metrics_df['tp'] + metrics_df['fn']).clip(lower=1)
metrics_df['n_consecutive_frames'] = N_CONSECUTIVE_FRAMES
metrics_df['source_log'] = str(LOG_PATH)
metrics_df['metric_mode'] = frame_level['mode'] if not gt_intervals else 'supervised_events'
display(metrics_df)
print('Predicted frame intervals:', len(pred_frame_intervals))
print('Predicted event intervals:', len(pred_event_intervals))

## 6. Save Event-vs-Frame Metrics

The output CSV is written to `/kaggle/working/results/event_vs_frame_metrics.csv`.

In [ ]:
out_csv = RESULTS_DIR / 'event_vs_frame_metrics.csv'
out_json = RESULTS_DIR / 'event_vs_frame_metrics.json'
metrics_df.to_csv(out_csv, index=False)
out_json.write_text(metrics_df.to_json(orient='records', indent=2) + '\n')
print('Saved:', out_csv)
print('Saved:', out_json)